In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/home/admin1/Downloads/NER dataset.csv", encoding="latin1")

In [3]:
df

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O
...,...,...,...,...
1048570,NaN,they,PRP,O
1048571,NaN,responded,VBD,O
1048572,NaN,to,TO,O
1048573,NaN,the,DT,O


In [4]:
df = df[['Sentence #', 'Word', 'POS']]

In [5]:
df.isnull().sum()

Sentence #    1000616
Word                0
POS                 0
dtype: int64

In [6]:
df['Sentence #'] = df['Sentence #'].ffill()

/tmp/ipykernel_6934/1121252816.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Sentence #'] = df['Sentence #'].ffill()


In [7]:
df['Word'] = df['Word'].bfill()

/tmp/ipykernel_6934/187573118.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Word'] = df['Word'].bfill()


In [8]:
sentences = df.groupby('Sentence #')['Word'].apply(list).values
pos_tags = df.groupby('Sentence #')['POS'].apply(list).values

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer

2026-04-15 10:40:36.103389: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-15 10:40:36.125685: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-15 10:40:36.974769: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [15]:
# words
word_tokenizer = Tokenizer()
word_tokenizer.fit_on_texts(sentences)
x = word_tokenizer.texts_to_sequences(sentences)

In [16]:
# tags
tag_tokenizer = Tokenizer()
tag_tokenizer.fit_on_texts(pos_tags)
y = tag_tokenizer.texts_to_sequences(pos_tags)

In [17]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [18]:
MAX_LEN = 100

In [19]:
x = pad_sequences(x, maxlen=MAX_LEN, padding='post')
y = pad_sequences(y, maxlen=MAX_LEN, padding='post')

In [20]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [21]:
from tensorflow.keras.models import Sequential

In [22]:
from tensorflow.keras.layers import Embedding, LSTM, Dense, TimeDistributed

In [23]:
model = Sequential()

In [24]:
model.add(Embedding(input_dim=len(word_tokenizer.word_index)+1, output_dim=32))
model.add(LSTM(32, return_sequences=True))
model.add(TimeDistributed(Dense(len(tag_tokenizer.word_index)+1, activation='softmax')))

In [25]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [26]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.fit(x_train, y_train, epochs=3, batch_size=32)

Epoch 1/3
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 25s 19ms/step - accuracy: 0.8583 - loss: 0.7442
Epoch 2/3
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 23s 19ms/step - accuracy: 0.9860 - loss: 0.0617
Epoch 3/3
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 23s 19ms/step - accuracy: 0.9915 - loss: 0.0311


In [28]:
loss, acc = model.evaluate(x_test, y_test)
print("Accuracy:", acc)

300/300 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9902 - loss: 0.0326
Accuracy: 0.990225613117218


In [29]:
import numpy as np

In [30]:
sentence = sentences[0]

seq = pad_sequences(
    word_tokenizer.texts_to_sequences([sentence]),
    maxlen=MAX_LEN,
    padding='post'
)

pred = np.argmax(model.predict(seq), axis=-1)[0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 413ms/step


In [31]:
index_to_tag = {v:k for k,v in tag_tokenizer.word_index.items()}

In [32]:
print("\nPrediction:")
for w, p in zip(sentence, pred):
    print(w, "->", index_to_tag.get(p))


Prediction:
Thousands -> nns
of -> in
demonstrators -> nns
have -> vbp
marched -> vbn
through -> in
London -> nnp
to -> to
protest -> vb
the -> dt
war -> nn
in -> in
Iraq -> nnp
and -> cc
demand -> nn
the -> dt
withdrawal -> nn
of -> in
British -> jj
troops -> nns
from -> in
that -> in
country -> nn
. -> .
